<a href="https://colab.research.google.com/github/Decoding-Data-Science/CommunityWorkshops/blob/main/bootcampoct26/notebooks/Final_Day2_02_RAG_with_LlamaIndex_Gemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 2 · Lab 2 · RAG with LlamaIndex: chat with your own documents

**Time:** about 60 minutes  |  **Runs in:** Google Colab  |  **Needs:** your Google Gemini key + 3 or 4 document files

### What you will do
1. Upload 3 or 4 documents into a folder.
2. Set the LlamaIndex **Settings**: model, embedding model, chunk size, overlap, top-k.
3. Look at the **chunks** and the **embedding** (the list of numbers).
4. Build a **vector store index** and run a **semantic search** with top_k = 3.
5. Ask questions and check the **sources**.
6. Wrap it in a **Gradio** app.

**RAG = Retrieval-Augmented Generation:** first *retrieve* the right pieces of your documents, then let the model *generate* an answer from them.

> All data is synthetic training data. The AI supports a person; a qualified person decides.

> ### Gemini version: what is different from the OpenAI notebook
> Only these places changed. Everything else is identical.
>
> | Where | OpenAI version | Gemini version |
> |---|---|---|
> | Key | `OPENAI_API_KEY`, from platform.openai.com | `GOOGLE_API_KEY`, from aistudio.google.com/apikey |
> | Install (Step 2) | `llama-index-llms-openai`, `llama-index-embeddings-openai` | `llama-index-llms-google-genai`, `llama-index-embeddings-google-genai` |
> | LLM (Step 4) | `OpenAI(model="gpt-5.4-mini", reasoning_effort="low")` | `GoogleGenAI(model="gemini-3.8-flash")` |
> | Embeddings (Step 4) | `OpenAIEmbedding(model="text-embedding-3-small")`, 1,536 numbers | `GoogleGenAIEmbedding(model_name="gemini-embedding-001")`, 3,072 numbers |
> | Gradio (Step 13) | plain | two extra lines at the top to avoid the Colab asyncio error |

## Step 1. Add your Gemini key to Colab Secrets (one time only)

1. Get a free key at **https://aistudio.google.com/apikey** (sign in with your Google account, click **Create API key**, copy it).
2. In Colab, click the **key icon** in the left sidebar, then **Add new secret**.
3. Name: `GOOGLE_API_KEY`. Value: paste your key.
4. Switch **Notebook access** ON.

Never type your key into a code cell, and never share a notebook that contains your key.

## Step 2. Install LlamaIndex (about 60 seconds)

In [1]:
# llama-index-core                  : the RAG framework
# llama-index-llms-google-genai     : lets LlamaIndex use Google Gemini chat models
# llama-index-embeddings-google-genai : lets LlamaIndex use Google Gemini embedding models
# llama-index-readers-file          : reads .docx and .pdf files
!pip install -q llama-index-core llama-index-llms-google-genai llama-index-embeddings-google-genai llama-index-readers-file docx2txt pypdf gradio

In [2]:
# Read the key from Colab Secrets and put it where the Google library looks for it.
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("gemini3")   # must match the secret name in Colab
print("Key loaded. It is", len(os.environ["GOOGLE_API_KEY"]), "characters long (we never print the key itself).")

Key loaded. It is 53 characters long (we never print the key itself).


## Step 3. Upload your documents

Run the cell, click **Choose files**, and select **3 or 4 files** (.docx, .pdf, .txt or .md). Use the files from the `rag_documents` folder in the Day 2 pack, or your own non-confidential documents.

*If the button does not appear:* open the **Files** panel (folder icon on the left), and drag the files into the `my_documents` folder.

In [3]:
import os, shutil
from google.colab import files

DOCS_FOLDER = "/content/my_documents"          # the folder that will hold your documents
os.makedirs(DOCS_FOLDER, exist_ok=True)         # create it (no error if it already exists)

uploaded = files.upload()                       # opens the "Choose files" window

for file_name in uploaded:                      # move every uploaded file into the folder
    shutil.move(file_name, os.path.join(DOCS_FOLDER, file_name))

print("Files in", DOCS_FOLDER)
for f in sorted(os.listdir(DOCS_FOLDER)):
    print("  -", f)

Saving DDS_Leave_Policy_v2.pdf to DDS_Leave_Policy_v2.pdf
Files in /content/my_documents
  - DDS_Leave_Policy_v2.pdf


## Step 4. The Settings: the five choices that shape your RAG system

| Setting | Our value | What it means |
|---|---|---|
| `Settings.llm` | gemini-3.8-flash | the model that writes the final answer |
| `Settings.embed_model` | gemini-embedding-001 | turns text into a vector of **3,072 numbers** |
| `Settings.chunk_size` | 400 tokens | how big each piece of a document is |
| `Settings.chunk_overlap` | 50 tokens | how much neighbouring chunks share, so no sentence is cut in half |
| `TOP_K` (similarity_top_k) | 3 | how many chunks we retrieve for each question |

In [6]:
from llama_index.core import Settings
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding

# The system prompt: the rules the model follows when it writes the answer.
SYSTEM_PROMPT = (
    '''
    You are an HR assistant for Decoding Data Science.

Answer questions using ONLY the attached HR policy.

Question:
How many annual leave days do employees receive?

Also mention the relevant policy section.

Be polite

if you cannot answer then request and send email to hr@decodingdatascience.com
'''
)

# 1. The LLM: writes the answer
#gemini-3.5-flash-lite - for free tier
Settings.llm = GoogleGenAI(
    model="gemini-3.8-flash",   # if this name is not available to you, change only this string
    system_prompt=SYSTEM_PROMPT,
)

# 2. The embedding model: turns text into vectors (lists of numbers)
Settings.embed_model = GoogleGenAIEmbedding(model_name="gemini-embedding-001")

# 3 and 4. Chunking
Settings.chunk_size = 300       # tokens per chunk
Settings.chunk_overlap = 40     # tokens shared between neighbouring chunks

# 5. How many chunks to retrieve per question
TOP_K = 3

print("LLM          :", Settings.llm.model)
print("Embedding    :", Settings.embed_model.model_name)
print("Chunk size   :", Settings.chunk_size, "tokens")
print("Chunk overlap:", Settings.chunk_overlap, "tokens")
print("Top-k        :", TOP_K)

LLM          : gemini-3.8-flash
Embedding    : gemini-embedding-001
Chunk size   : 300 tokens
Chunk overlap: 40 tokens
Top-k        : 3


## Step 5. Load the documents

In [7]:
from llama_index.core import SimpleDirectoryReader

documents = SimpleDirectoryReader(DOCS_FOLDER).load_data()    # reads every file in the folder

print("Loaded", len(documents), "document pages/parts")
for d in documents:
    print(f"  {d.metadata['file_name'][:60]:<60} {len(d.text):>6} characters")

Loaded 4 document pages/parts
  DDS_Leave_Policy_v2.pdf                                        3638 characters
  DDS_Leave_Policy_v2.pdf                                        2941 characters
  DDS_Leave_Policy_v2.pdf                                        3509 characters
  DDS_Leave_Policy_v2.pdf                                         978 characters


## Step 6. Chunking: cut the documents into pieces

Why chunk? A whole document is too big and too mixed to search well. Small pieces let us find **exactly** the paragraph that answers the question.

- **Too small** (e.g. 100 tokens): pieces lose their meaning.
- **Too big** (e.g. 2,000 tokens): the search is vague and you send lots of extra text to the model.
- **Overlap**: the end of one chunk is repeated at the start of the next, so an idea that crosses the border is not lost.

In [8]:
from llama_index.core.node_parser import SentenceSplitter

# Same settings as above. SentenceSplitter tries not to cut sentences in half.
splitter = SentenceSplitter(chunk_size=Settings.chunk_size, chunk_overlap=Settings.chunk_overlap)
nodes = splitter.get_nodes_from_documents(documents)       # a "node" = one chunk + its metadata

print("Number of chunks:", len(nodes))
print("\n----- Chunk 1 -----\n", nodes[0].get_content()[:700])
print("\n----- Chunk 2 -----\n", nodes[1].get_content()[:700])

Number of chunks: 12

----- Chunk 1 -----
 Leave Policy - Decoding Data Science
Document ID: DDS-HR-POL-001 | Version 2.0 | Effective date: 1 July 2026
Internal - Synthetic demo document for AI training purposes. Not legal advice.
This policy defines all leave entitlements at DDS, how leave accrues, how to request it, and how requests are
approved. It replaces Leave Policy v1 (March 2026) in full. Where UAE Federal Decree-Law No. 33 of 2021 (the UAE
Labour Law) provides a greater benefit, the law prevails. Note: this is a synthetic, training-friendly policy created for
demos, onboarding simulations, and HR-chatbot prototypes. It is not legal advice.
1. Leave Entitlements at a Glance
Annual leave: 22 working days per year, full pay, 7

----- Chunk 2 -----
 Maternity leave: 60 calendar days, full pay, 30 calendar
days' notice before expected date, medical documentation required. Paternity leave: 5 working days within 6 months
of birth, full pay, 30 calendar days' notice where foreseeable

In [9]:
# Compare chunk sizes (this costs nothing, it is only cutting text).
import pandas as pd

rows = []
for size in [200, 400, 800]:
    n = SentenceSplitter(chunk_size=size, chunk_overlap=50).get_nodes_from_documents(documents)
    rows.append({"chunk_size (tokens)": size, "chunk_overlap": 50, "number of chunks": len(n)})

pd.DataFrame(rows)

,chunk_size (tokens),chunk_overlap,number of chunks
0,200,50,21
1,400,50,9
2,800,50,5


**Discuss:** with chunk size 200 you get many small chunks; with 800 you get a few big ones. Which would you choose for short procedures? For long reports?

## Step 7. Embeddings: text becomes numbers

An **embedding** is a list of numbers that captures the *meaning* of a text. Texts with similar meaning get similar numbers, even if they use different words.

In [10]:
import numpy as np

vector = Settings.embed_model.get_text_embedding("Leave policy")

print("Dimensions:", len(vector))                 # gemini-embedding-001 -> 3072 numbers
print("First 8 numbers:", np.round(vector[:8], 4))

Dimensions: 3072
First 8 numbers: [-0.0157  0.0346 -0.0079 -0.0544  0.0012  0.008   0.0077  0.0083]


In [11]:
# Semantic similarity: which sentence is closest in MEANING to the question?
def cosine(a, b):
    a, b = np.array(a), np.array(b)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))   # 1.0 = same meaning

question = "Can I take time off for my wedding?"
sentences = [
    "Does the company offer paid leave for marriages?",                 # same meaning, different words
    "Employees are entitled to annual leave accrued monthly.",          # related
    "All company laptops must be returned upon resignation.",            # unrelated
]

q_vec = Settings.embed_model.get_text_embedding(question)
for s in sentences:
    print(f"{cosine(q_vec, Settings.embed_model.get_text_embedding(s)):.3f}  {s}")

0.918  Does the company offer paid leave for marriages?
0.818  Employees are entitled to annual leave accrued monthly.
0.752  All company laptops must be returned upon resignation.


Notice: the question says *"shaking"*, the best sentence says *"vibration"*. A keyword search would miss it. A **semantic search** finds it, because the meanings are close.

## Step 8. Build the vector store index

`VectorStoreIndex` embeds **every chunk** and stores the vectors together with the text and metadata (file name). That store is our **vector store**. Today it lives in memory; in production it would be a vector database.

In [12]:
from llama_index.core import VectorStoreIndex

index = VectorStoreIndex(nodes)        # embeds every chunk with Settings.embed_model and stores it

print("Index ready with", len(nodes), "chunks, each stored as a vector of", len(vector), "numbers.")

Index ready with 12 chunks, each stored as a vector of 3072 numbers.


## Step 9. Semantic search with top_k = 3

The **retriever** embeds your question, compares it with every chunk vector, and returns the **3 most similar chunks** with a similarity score. No answer is written yet: this is only the search.

In [13]:
retriever = index.as_retriever(similarity_top_k=TOP_K)

question = "how many annual leaves do i get?"
results = retriever.retrieve(question)

for rank, r in enumerate(results, start=1):
    print(f"#{rank}  score={r.score:.3f}  file={r.node.metadata['file_name']}")
    print("    ", r.node.get_content()[:250].replace("\n", " "), "...\n")

#1  score=0.717  file=DDS_Leave_Policy_v2.pdf
     Unpaid leave: up to 30 calendar days per year, case-by-case, unpaid, 30 calendar days' notice, manager and HR approval required. Public holidays: per the official UAE calendar, full pay, not deducted from annual leave. 2. Definitions Leave year: 1 Ja ...

#2  score=0.688  file=DDS_Leave_Policy_v2.pdf
     3.3 Booking limits  The maximum single block of annual leave is 15 consecutive working days. Longer blocks require CEO approval.  At least 5 of the 22 days must be taken as one continuous block of 5 working days each year, to support genuine rest.  ...

#3  score=0.685  file=DDS_Leave_Policy_v2.pdf
     Maternity leave: 60 calendar days, full pay, 30 calendar days' notice before expected date, medical documentation required. Paternity leave: 5 working days within 6 months of birth, full pay, 30 calendar days' notice where foreseeable, birth certific ...



## Step 10. Ask questions: retrieve + generate

The **query engine** does both steps: retrieve the top 3 chunks, put them into the prompt, and let the model write the answer. We also print the **sources** so a person can check.

In [16]:
import nest_asyncio
nest_asyncio.apply()

query_engine = index.as_query_engine(similarity_top_k=TOP_K)

def ask_docs(question):
    response = query_engine.query(question)
    print("QUESTION:", question)
    print("\nANSWER:\n", response.response)
    print("\nSOURCES:")
    for s in response.source_nodes:
        print(f"  - {s.node.metadata['file_name']}  (score {s.score:.3f})")
    print("-" * 80)

ask_docs("how many annual leaves do i get?")

QUESTION: how many annual leaves do i get?

ANSWER:
 Full-time employees are entitled to 22 working days of paid annual leave per leave year. Part-time employees receive a pro-rata entitlement based on their contracted hours, and employees who join or leave mid-year receive a pro-rata entitlement for that year.

This can be found under **Section 3.1 (Entitlement)** of the Annual Leave policy. 

If you have any further questions or need additional assistance, please feel free to contact hr@decodingdatascience.com.

SOURCES:
  - DDS_Leave_Policy_v2.pdf  (score 0.717)
  - DDS_Leave_Policy_v2.pdf  (score 0.688)
  - DDS_Leave_Policy_v2.pdf  (score 0.685)
--------------------------------------------------------------------------------


In [17]:
# Ask more questions. Replace them with your own.
ask_docs("How many days off?")
ask_docs("how many sick leaves do we get")
ask_docs("can we carry forward annual leaves")      # not in the documents: it should say so

QUESTION: How many days off?

ANSWER:
 Hello! Full-time employees receive 22 working days of paid annual leave per leave year. Part-time employees, as well as employees who join or leave mid-year, receive a pro-rata entitlement based on their contracted hours or service.

This information is outlined in **Section 3.1 (Entitlement)** of the leave policy. 

If you have any further questions or need additional assistance, please feel free to send an email to hr@decodingdatascience.com.

SOURCES:
  - DDS_Leave_Policy_v2.pdf  (score 0.697)
  - DDS_Leave_Policy_v2.pdf  (score 0.689)
  - DDS_Leave_Policy_v2.pdf  (score 0.678)
--------------------------------------------------------------------------------
QUESTION: how many sick leaves do we get

ANSWER:
 Employees receive 15 working days of fully paid sick leave per leave year, which becomes available after completing the probation period, as outlined in Section 5.1 (and Section 1). 

If an illness continues beyond the initial 15 fully paid 

## Step 11. With RAG vs without RAG

Same model, same question. Left: no documents. Right: with the retrieved chunks.

In [18]:
q = "can we carry forward annual leaves?"

print("===== WITHOUT RAG (model only) =====")
print(Settings.llm.complete(q).text[:800])

print("\n===== WITH RAG (top 3 chunks from your documents) =====")
print(query_engine.query(q).response)

===== WITHOUT RAG (model only) =====
Whether you can carry forward unused annual leave depends on **your company’s internal policy** and the **labor laws of your country/state**. 

Here is how it generally works across most workplaces:

---

### 1. Common Company Practices
* **Capped Carry-Forward (Most Common):** Many companies allow you to carry over a limited number of days (e.g., up to 5 or 10 days) into the next year.
* **Time-Bound Carry-Forward:** Some companies allow carry-over, but you must use those days within a specific window (e.g., within the first 3 or 6 months of the new year), or they expire.
* **"Use-It-or-Lose-It":** In jurisdictions where it is legal, some companies enforce a strict policy where any unused leave at the end of the year is forfeited without compensation.
* **Leave Encashment:** Instead of ca

===== WITH RAG (top 3 chunks from your documents) =====
Yes, you can carry forward your unused annual leave. 

A maximum of 10 unused annual-leave days may be ca

## Step 12. What does top_k change?

In [19]:
for k in [1, 3, 5]:
    r = index.as_retriever(similarity_top_k=k).retrieve(q)
    print(f"top_k={k}:", [f"{x.node.metadata['file_name'][:25]} ({x.score:.2f})" for x in r])

top_k=1: ['DDS_Leave_Policy_v2.pdf (0.70)']
top_k=3: ['DDS_Leave_Policy_v2.pdf (0.70)', 'DDS_Leave_Policy_v2.pdf (0.68)', 'DDS_Leave_Policy_v2.pdf (0.65)']
top_k=5: ['DDS_Leave_Policy_v2.pdf (0.70)', 'DDS_Leave_Policy_v2.pdf (0.68)', 'DDS_Leave_Policy_v2.pdf (0.65)', 'DDS_Leave_Policy_v2.pdf (0.65)', 'DDS_Leave_Policy_v2.pdf (0.64)']


**Rule of thumb:** top_k too small may miss the evidence; too big adds noise and cost. 3 to 5 is a good start.

## Step 13. Gradio app: chat with your documents

In [20]:
import asyncio, uvicorn.server
uvicorn.server.asyncio_run = asyncio.runners.run   # fixes the Colab "loop_factory" Gradio error

import gradio as gr

def rag_app(question):
    if not question.strip():
        return "Please type a question.", ""
    response = query_engine.query(question)
    sources = "\n".join(f"- **{s.node.metadata['file_name']}** (score {s.score:.2f})"
                        for s in response.source_nodes)
    return response.response, sources

demo = gr.Interface(
    fn=rag_app,
    inputs=gr.Textbox(label="Ask a question about your documents", lines=2),
    outputs=[gr.Markdown(label="Answer"), gr.Markdown(label="Sources (top 3 chunks)")],
    title="Document Assistant (RAG)",
    description=f"Answers come only from the files you uploaded. chunk_size={Settings.chunk_size}, overlap={Settings.chunk_overlap}, top_k={TOP_K}.",
    examples=[["can we carry forward annual leaves"],
              ["How many sick leaves?"],
              ["Process to take annual leave?"]],
)

demo.launch(share=True)

/usr/local/lib/python3.13/dist-packages/fastapi/applications.py:7: RuntimeWarning: coroutine 'prepare_chat_params' was never awaited
  from fastapi import routing


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f8da25cbbd241d91db.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Exercises

1. Change `Settings.chunk_size` to **200**, re-run from Step 6, and ask the same questions. Better or worse?
2. Change `TOP_K` to **1** and then **5**. When does the answer miss information?
3. Upload a **different** set of files (for example the security documents) and build a new assistant.
4. Edit `SYSTEM_PROMPT` so every answer ends with *"Please verify in the source document."*

### Key takeaways
- RAG has two phases. **Index**: load, chunk, embed, store. **Query**: embed the question, find top-k, answer with sources.
- Chunk size and overlap decide what the search can find.
- Embeddings capture meaning, so semantic search finds "shaking" when the document says "vibration".
- Always look at the sources. The model can still be wrong.